# Semana 20 — Fine-tuning: cuándo, cómo y qué alternativas hay

**Objetivo:** construir un clasificador de documentos del sector agroalimentario
usando fine-tuning de `gpt-4o-mini`, compararlo contra few-shot prompting,
y decidir cuál es la mejor opción para el caso de uso.

**Caso de uso:** clasificar automáticamente documentos que llegan a una empresa
del sector pesquero (informes de captura, certificaciones, normativa, facturas...)
para enrutarlos al departamento correcto.

**Al finalizar tendrás:**
- Un dataset de fine-tuning en formato `.jsonl`
- Un modelo fine-tuneado con `gpt-4o-mini`
- Una comparación: fine-tuned vs few-shot vs base model
- La decisión justificada sobre qué usar en producción

**Coste estimado:**
- Dataset: $0.00 (generamos ejemplos sintéticos con GPT)
- Fine-tuning (200 ejemplos × 3 épocas): ~$0.10–0.20
- Evaluación: ~$0.01
- **Total: ~$0.15–0.25**

## Celda 0 — Setup

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai", "python-dotenv"])

import os, json, time, random
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
assert OPENAI_API_KEY, "Define OPENAI_API_KEY en tu .env"

client = OpenAI(api_key=OPENAI_API_KEY)

DATA_DIR = Path("ft_data")
DATA_DIR.mkdir(exist_ok=True)

MODELO_BASE = "gpt-4o-mini"
random.seed(42)

print(f"✓ OpenAI SDK listo")
print(f"✓ Directorio de datos: {DATA_DIR.resolve()}")

## Celda 1 — Definir la tarea y las categorías

Antes de construir el dataset, define exactamente qué clasifica el modelo
y qué formato debe devolver.

In [ ]:
# Taxonomía de documentos del sector pesquero/agroalimentario
CATEGORIAS = {
    "informes_capturas": {
        "descripcion": "Informes de capturas, cuotas, volúmenes de pesca",
        "ejemplos_titulo": [
            "Informe capturas merluza caladero Atlántico Norte Q3 2025",
            "Cuota anual bacalao flota gallega 2025",
            "Estadísticas desembarco puerto Vigo septiembre 2025",
        ]
    },
    "certificaciones": {
        "descripcion": "Certificados de calidad, auditorías, acreditaciones",
        "ejemplos_titulo": [
            "Certificado IFS Food planta procesado Vigo 2025",
            "Auditoría BRC almacén frigorífico A Coruña",
            "Renovación ISO 22000 conservera Galicia",
        ]
    },
    "normativa": {
        "descripcion": "Reglamentos, directivas, legislación aplicable",
        "ejemplos_titulo": [
            "Reglamento UE 2023/1115 debida diligencia cadena suministro",
            "Real Decreto etiquetado productos pesqueros 2024",
            "Directiva tallas mínimas captura especies demersales",
        ]
    },
    "analisis_mercado": {
        "descripcion": "Informes de precios, tendencias de mercado, competencia",
        "ejemplos_titulo": [
            "Precios lonja Vigo semana 42 2025",
            "Análisis importaciones pescado congelado España Q4 2025",
            "Informe tendencias consumo productos del mar Europa 2025",
        ]
    },
    "documentacion_tecnica": {
        "descripcion": "Fichas técnicas, especificaciones de producto, protocolos",
        "ejemplos_titulo": [
            "Ficha técnica conserva mejillón en escabeche lata 120g",
            "Protocolo HACCP línea cocción langostino",
            "Especificaciones microbiológicas producto fresco refrigerado",
        ]
    },
}

SYSTEM_PROMPT = """Eres un clasificador de documentos para el sector pesquero y agroalimentario.
Clasifica el título del documento en una de estas categorías exactas:
- informes_capturas
- certificaciones
- normativa
- analisis_mercado
- documentacion_tecnica

Responde SOLO con el nombre de la categoría, sin explicación."""

print(f"Taxonomía: {len(CATEGORIAS)} categorías")
for cat, info in CATEGORIAS.items():
    print(f"  · {cat}: {info['descripcion']}")

## Celda 2 — Generar el dataset con destilación desde GPT-4o

Usamos GPT-4o para generar ejemplos de entrenamiento sintéticos.
Este es el patrón de **destilación**: usas un modelo más caro para crear
el dataset y luego entrenas un modelo más barato.

In [ ]:
CACHE_DATASET = DATA_DIR / "dataset_raw.json"

def generar_titulos_categoria(categoria: str, descripcion: str, n: int = 25) -> list[str]:
    """Genera títulos de documentos sintéticos para una categoría."""
    resp = client.chat.completions.create(
        model="gpt-4o-mini",   # usamos mini para abaratar la generación
        messages=[{
            "role": "user",
            "content": (
                f"Genera {n} títulos realistas de documentos de la categoría '{categoria}' "
                f"({descripcion}) para una empresa del sector pesquero y agroalimentario gallego.\n"
                f"Los títulos deben ser variados: distintos productos, períodos, entidades.\n"
                f"Devuelve SOLO los títulos, uno por línea, sin numeración ni explicación."
            )
        }],
        temperature=0.9,
        max_tokens=800,
    )
    lineas = resp.choices[0].message.content.strip().split("\n")
    return [l.strip() for l in lineas if l.strip()][:n]


if CACHE_DATASET.exists():
    dataset_raw = json.loads(CACHE_DATASET.read_text())
    print(f"✓ Dataset cargado desde caché ({sum(len(v) for v in dataset_raw.values())} ejemplos)")
else:
    print("Generando ejemplos por categoría...")
    dataset_raw = {}
    for cat, info in CATEGORIAS.items():
        titulos = generar_titulos_categoria(cat, info["descripcion"], n=40)
        dataset_raw[cat] = titulos
        print(f"  ✓ {cat}: {len(titulos)} títulos generados")

    CACHE_DATASET.write_text(json.dumps(dataset_raw, ensure_ascii=False, indent=2))
    print(f"\n✓ Dataset guardado en caché")

total = sum(len(v) for v in dataset_raw.values())
print(f"\nTotal: {total} ejemplos | {total // len(CATEGORIAS)} por categoría (media)")

## Celda 3 — Construir train/eval splits y convertir a JSONL

In [ ]:
def construir_mensaje(titulo: str, categoria: str) -> dict:
    """Convierte un ejemplo al formato de fine-tuning de OpenAI."""
    return {
        "messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": f"Clasificar: '{titulo}'"},
            {"role": "assistant", "content": categoria},
        ]
    }


# Construye todos los ejemplos
todos = []
for cat, titulos in dataset_raw.items():
    for titulo in titulos:
        todos.append((titulo, cat))

random.shuffle(todos)

# Split 80/20 estratificado por categoría
train_ejemplos, eval_ejemplos = [], []
for cat in CATEGORIAS:
    cat_items = [(t, c) for t, c in todos if c == cat]
    n_train   = int(len(cat_items) * 0.8)
    train_ejemplos.extend(cat_items[:n_train])
    eval_ejemplos.extend(cat_items[n_train:])

random.shuffle(train_ejemplos)
random.shuffle(eval_ejemplos)

# Guarda los archivos JSONL
TRAIN_FILE = DATA_DIR / "train.jsonl"
EVAL_FILE  = DATA_DIR / "eval.jsonl"

for ruta, ejemplos in [(TRAIN_FILE, train_ejemplos), (EVAL_FILE, eval_ejemplos)]:
    with ruta.open("w", encoding="utf-8") as f:
        for titulo, cat in ejemplos:
            f.write(json.dumps(construir_mensaje(titulo, cat), ensure_ascii=False) + "\n")

print(f"✓ Train: {len(train_ejemplos)} ejemplos → {TRAIN_FILE}")
print(f"✓ Eval:  {len(eval_ejemplos)} ejemplos → {EVAL_FILE}")
print()

# Muestra distribución
from collections import Counter
dist_train = Counter(c for _, c in train_ejemplos)
print("Distribución en train:")
for cat, n in sorted(dist_train.items()):
    print(f"  {cat:<30} {n:>3} ejemplos")

# Muestra 2 ejemplos del JSONL
print("\nEjemplos del archivo JSONL (primeros 2):")
with TRAIN_FILE.open() as f:
    for i, linea in enumerate(f):
        if i >= 2: break
        ej = json.loads(linea)
        print(f"  User:      {ej['messages'][1]['content']}")
        print(f"  Assistant: {ej['messages'][2]['content']}")
        print()

## Celda 4 — Baseline: evaluar el modelo base sin fine-tuning

Antes de hacer fine-tuning, mide cuánto rinde el modelo base.
Esto es tu línea de base para saber si el fine-tuning mejora algo.

In [ ]:
def clasificar(titulo: str, modelo: str, few_shot_ejemplos: list[dict] | None = None) -> str:
    """Clasifica un título. Si few_shot_ejemplos, los añade al contexto."""
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    if few_shot_ejemplos:
        for ej in few_shot_ejemplos:
            messages.append({"role": "user",      "content": f"Clasificar: '{ej['titulo']}'" })
            messages.append({"role": "assistant", "content": ej["categoria"]})

    messages.append({"role": "user", "content": f"Clasificar: '{titulo}'"})

    resp = client.chat.completions.create(
        model=modelo,
        messages=messages,
        temperature=0.0,
        max_tokens=30,
    )
    pred = resp.choices[0].message.content.strip().lower()
    # Normaliza: extrae solo la categoría si el modelo devuelve texto extra
    for cat in CATEGORIAS:
        if cat in pred:
            return cat
    return pred


def evaluar_modelo(modelo: str, eval_set: list[tuple],
                   few_shot: list[dict] | None = None,
                   max_ejemplos: int = 50) -> dict:
    """Evalúa el modelo sobre el eval set. Devuelve accuracy y métricas por clase."""
    subset = eval_set[:max_ejemplos]
    correctas, totales = 0, 0
    por_clase = {cat: {"tp": 0, "total": 0} for cat in CATEGORIAS}

    for titulo, cat_real in subset:
        pred = clasificar(titulo, modelo, few_shot)
        correctas += pred == cat_real
        totales   += 1
        por_clase[cat_real]["total"] += 1
        if pred == cat_real:
            por_clase[cat_real]["tp"] += 1

    accuracy = correctas / totales if totales else 0
    return {
        "modelo":    modelo,
        "accuracy":  round(accuracy, 3),
        "correctas": correctas,
        "totales":   totales,
        "por_clase": {
            cat: round(v["tp"] / v["total"], 2) if v["total"] else 0
            for cat, v in por_clase.items()
        },
    }


# Evalúa el modelo base (sin ejemplos en el prompt)
print("Evaluando modelo base (zero-shot)...")
resultado_base = evaluar_modelo(MODELO_BASE, eval_ejemplos, max_ejemplos=30)

print(f"\nModelo base zero-shot:")
print(f"  Accuracy: {resultado_base['accuracy']:.1%}  ({resultado_base['correctas']}/{resultado_base['totales']})")
print("  Por clase:")
for cat, acc in resultado_base["por_clase"].items():
    barra = "█" * int(acc * 20) + "░" * (20 - int(acc * 20))
    print(f"    {cat:<30} {barra} {acc:.0%}")

## Celda 5 — Few-shot baseline: ¿cuánto mejora con ejemplos en el prompt?

In [ ]:
# Selecciona 2 ejemplos por categoría (10 en total) del train set para few-shot
few_shot_ejemplos = []
for cat in CATEGORIAS:
    cat_train = [(t, c) for t, c in train_ejemplos if c == cat]
    for titulo, _ in cat_train[:2]:
        few_shot_ejemplos.append({"titulo": titulo, "categoria": cat})

print(f"Few-shot: {len(few_shot_ejemplos)} ejemplos ({len(few_shot_ejemplos)//len(CATEGORIAS)} por clase)")
print()

print("Evaluando modelo base (few-shot 10 ejemplos)...")
resultado_fewshot = evaluar_modelo(
    MODELO_BASE, eval_ejemplos,
    few_shot=few_shot_ejemplos,
    max_ejemplos=30
)

print(f"\nModelo base few-shot (10 ejemplos):")
print(f"  Accuracy: {resultado_fewshot['accuracy']:.1%}  ({resultado_fewshot['correctas']}/{resultado_fewshot['totales']})")
print(f"  Mejora vs zero-shot: +{(resultado_fewshot['accuracy']-resultado_base['accuracy'])*100:.1f}pp")

## Celda 6 — Lanzar el fine-tuning job

Sube el dataset y lanza el job. El entrenamiento tarda entre 15 minutos y 1 hora
según el tamaño del dataset y la carga de OpenAI.

In [ ]:
JOB_CACHE = DATA_DIR / "ft_job.json"

if JOB_CACHE.exists():
    job_info = json.loads(JOB_CACHE.read_text())
    print(f"✓ Job existente cargado desde caché")
    print(f"  Job ID:  {job_info['job_id']}")
    print(f"  Status:  {job_info.get('status', 'desconocido')}")
    if job_info.get("modelo_ft"):
        print(f"  Modelo:  {job_info['modelo_ft']}")
else:
    # 1. Sube el archivo de entrenamiento
    print("Subiendo dataset de entrenamiento...")
    with TRAIN_FILE.open("rb") as f:
        upload = client.files.create(file=f, purpose="fine-tune")
    print(f"✓ Archivo subido: {upload.id}")

    # 2. Lanza el job
    print("Lanzando job de fine-tuning...")
    job = client.fine_tuning.jobs.create(
        training_file=upload.id,
        model=MODELO_BASE,
        hyperparameters={
            "n_epochs": 3,              # más épocas = más coste, más riesgo de overfitting
        },
        suffix="clasificador-pesquero",  # aparece en el nombre del modelo
    )

    job_info = {"job_id": job.id, "file_id": upload.id, "status": job.status}
    JOB_CACHE.write_text(json.dumps(job_info, indent=2))

    print(f"✓ Job lanzado: {job.id}")
    print(f"  Status inicial: {job.status}")
    print(f"  Estimated finish: {job.estimated_finish}")

print()
print("Puedes hacer polling del estado con la celda 7.")
print("O consultar en: https://platform.openai.com/finetune")

## Celda 7 — Polling: esperar a que el job termine

Ejecuta esta celda periódicamente hasta que el status sea `succeeded`.
Puede tardar 15–60 minutos.

In [ ]:
job_info = json.loads(JOB_CACHE.read_text())
job_id   = job_info["job_id"]

job = client.fine_tuning.jobs.retrieve(job_id)

print(f"Job ID:  {job.id}")
print(f"Status:  {job.status}")
print(f"Modelo:  {job.fine_tuned_model or '(pendiente)'}")
print()

# Muestra los últimos eventos del job
eventos = client.fine_tuning.jobs.list_events(job_id, limit=5)
print("Últimos eventos:")
for ev in reversed(list(eventos.data)):
    print(f"  [{ev.level}] {ev.message}")

# Guarda el modelo si ya está listo
if job.status == "succeeded" and job.fine_tuned_model:
    job_info["modelo_ft"] = job.fine_tuned_model
    job_info["status"]    = job.status
    JOB_CACHE.write_text(json.dumps(job_info, indent=2))
    print(f"\n✓ Modelo fine-tuneado listo: {job.fine_tuned_model}")
elif job.status == "failed":
    print(f"\n✗ Job fallido: {job.error}")
else:
    print(f"\n⏳ Job en curso — vuelve a ejecutar esta celda en unos minutos")

## Celda 8 — Evaluar el modelo fine-tuneado y comparar

Ejecuta esta celda solo cuando el job haya terminado (`status: succeeded`).

In [ ]:
job_info = json.loads(JOB_CACHE.read_text())
MODELO_FT = job_info.get("modelo_ft")

if not MODELO_FT:
    print("⚠  El modelo fine-tuneado no está listo todavía.")
    print("   Espera a que el job termine y vuelve a ejecutar.")
else:
    print(f"Evaluando modelo fine-tuneado: {MODELO_FT}")
    print("(puede tardar 1-2 minutos)\n")

    resultado_ft = evaluar_modelo(MODELO_FT, eval_ejemplos, max_ejemplos=50)

    # ── Tabla comparativa ─────────────────────────────────────────────────────
    print("=" * 65)
    print(" Comparativa de modelos")
    print("=" * 65)
    print(f"{'Modelo':<35} {'Accuracy':>10} {'Correctas':>12}")
    print("-" * 65)

    resultados = [
        ("Base zero-shot",       resultado_base),
        ("Base few-shot (10ej)", resultado_fewshot),
        ("Fine-tuned",           resultado_ft),
    ]
    for nombre, r in resultados:
        marca = " ←"
        print(f"  {nombre:<33} {r['accuracy']:>9.1%} {r['correctas']:>5}/{r['totales']:<5}")

    mejor = max(resultados, key=lambda x: x[1]["accuracy"])
    print(f"\n  Mejor modelo: {mejor[0]} ({mejor[1]['accuracy']:.1%})")

    # ── Análisis por clase ────────────────────────────────────────────────────
    print()
    print(f"{'Categoría':<30} {'Base':>8} {'Few-shot':>10} {'Fine-tuned':>12}")
    print("-" * 65)
    for cat in CATEGORIAS:
        b  = resultado_base["por_clase"].get(cat, 0)
        fs = resultado_fewshot["por_clase"].get(cat, 0)
        ft = resultado_ft["por_clase"].get(cat, 0)
        mejor_clase = max(b, fs, ft)
        marca_b  = " ◄" if b  == mejor_clase else ""
        marca_fs = " ◄" if fs == mejor_clase else ""
        marca_ft = " ◄" if ft == mejor_clase else ""
        print(f"  {cat:<28} {b:>7.0%} {fs:>9.0%}{marca_fs:<2} {ft:>10.0%}{marca_ft}")

## Celda 9 — Análisis de coste: ¿vale la pena el fine-tuning?

In [ ]:
# ── Parámetros del caso de uso ────────────────────────────────────────────────
TOKENS_POR_CLASIFICACION = 120   # system + user + respuesta
CLASIFICACIONES_DIA      = 500   # documentos que llegan al día
DIAS_MES                 = 22

# ── Costes de inferencia ──────────────────────────────────────────────────────
PRECIO_BASE_INPUT  = 0.15   # gpt-4o-mini, $/1M tokens
PRECIO_BASE_OUTPUT = 0.60
PRECIO_FT_INPUT    = 0.30   # modelo fine-tuneado cuesta 2× en inferencia
PRECIO_FT_OUTPUT   = 1.20

tokens_input  = int(TOKENS_POR_CLASIFICACION * 0.85)
tokens_output = int(TOKENS_POR_CLASIFICACION * 0.15)
n_mes         = CLASIFICACIONES_DIA * DIAS_MES

coste_base_mes = (tokens_input * PRECIO_BASE_INPUT + tokens_output * PRECIO_BASE_OUTPUT) / 1e6 * n_mes
coste_ft_mes   = (tokens_input * PRECIO_FT_INPUT   + tokens_output * PRECIO_FT_OUTPUT  ) / 1e6 * n_mes

# Few-shot añade tokens extra (~200 tokens por los 10 ejemplos)
coste_fewshot_mes = (
    (tokens_input + 200) * PRECIO_BASE_INPUT + tokens_output * PRECIO_BASE_OUTPUT
) / 1e6 * n_mes

# Coste del fine-tuning (una sola vez)
tokens_train  = len(train_ejemplos) * TOKENS_POR_CLASIFICACION
coste_ft_unico = tokens_train * 3 * 3.00 / 1e6  # 3 épocas, $3/1M tokens

print("=" * 55)
print(f" Análisis de coste ({CLASIFICACIONES_DIA} docs/día, {DIAS_MES} días/mes)")
print("=" * 55)
print(f"  Base zero-shot:    ${coste_base_mes:.3f}/mes")
print(f"  Base few-shot:     ${coste_fewshot_mes:.3f}/mes  (+{(coste_fewshot_mes-coste_base_mes)/coste_base_mes*100:.0f}% por tokens extra)")
print(f"  Fine-tuned:        ${coste_ft_mes:.3f}/mes  (2× precio inferencia)")
print(f"  Coste entrenamiento (único): ${coste_ft_unico:.3f}")
print()

# Break-even: ¿cuándo se amortiza el fine-tuning vs few-shot?
ahorro_mensual = coste_fewshot_mes - coste_ft_mes
if ahorro_mensual > 0:
    breakeven_meses = coste_ft_unico / ahorro_mensual
    print(f"  El fine-tuning es más barato que few-shot a partir del mes {breakeven_meses:.1f}")
else:
    print(f"  Few-shot es más barato que fine-tuned en este volumen")
    print(f"  (el fine-tuning solo compensa si mejora significativamente la accuracy)")

print()
print("Conclusión:")
if 'resultado_ft' in dir() and resultado_ft:
    mejora = resultado_ft['accuracy'] - resultado_fewshot['accuracy']
    if mejora > 0.05:
        print(f"  Fine-tuning mejora la accuracy +{mejora*100:.1f}pp sobre few-shot → recomendado")
    elif mejora > 0:
        print(f"  Fine-tuning mejora solo +{mejora*100:.1f}pp → evalúa si justifica el coste extra")
    else:
        print(f"  Few-shot supera o iguala al fine-tuned → usa few-shot (más simple)")
else:
    print("  (ejecuta la celda 8 para ver la recomendación con datos reales)")

## Reto — Análisis de errores del modelo fine-tuneado

El accuracy global no te dice dónde falla el modelo. Implementa un análisis de errores
que muestre los 5 errores más frecuentes y la matriz de confusión.

```python
def analizar_errores(modelo: str, eval_set: list[tuple], max_ej: int = 50) -> dict:
    """
    Devuelve:
    - confusion_matrix: dict[real][pred] = count
    - errores: lista de {titulo, real, predicho} ordenada por frecuencia del error
    """
    # TODO: implementar
    # 1. Para cada ejemplo, obtén la predicción
    # 2. Si es incorrecta, añádela a la lista de errores
    # 3. Construye la matriz de confusión
    pass


def imprimir_confusion_matrix(matriz: dict):
    """Imprime la matriz de confusión de forma legible."""
    # TODO: implementar
    # Pista: usa un formato de tabla con las categorías como filas y columnas
    pass
```

Preguntas para reflexionar tras el análisis:
- ¿Qué par de categorías se confunde más? ¿Tiene sentido semánticamente?
- ¿Los errores son ambigüedades reales (el humano también dudaría) o errores claros?
- ¿Cómo mejorarías el dataset para corregir los errores más frecuentes?

## Resumen de la semana 20

| Concepto | Dónde lo aplicaste |
|---------|-------------------|
| Árbol de decisión fine-tuning vs RAG vs few-shot | Dispensa + Celda 9 |
| Destilación: GPT-4o genera el dataset | Celda 2 |
| Formato JSONL para fine-tuning OpenAI | Celda 3 |
| Train/eval split estratificado | Celda 3 |
| Baseline zero-shot y few-shot | Celdas 4-5 |
| `fine_tuning.jobs.create()` + polling | Celdas 6-7 |
| Evaluación comparativa 3 modelos | Celda 8 |
| Análisis de coste y break-even | Celda 9 |
| Análisis de errores + matriz de confusión | Reto |

La semana 21 cubre **seguridad en sistemas LLM**: prompt injection, jailbreaks,
guardrails y cómo proteger una API de producción frente a uso malicioso.